### 260914 학습내용

#### 상세 학습 내용

##### 1. CSV 데이터를 `df`라는 변수에 저장하는 이유

**질문**

CSV 데이터를 읽은 결과를 `df`라는 변수에 저장하는 이유는 무엇인가?

**답변**

`pd.read_csv()`는 CSV 파일을 읽어 pandas의 `DataFrame`으로 변환한다. 이 결과를 `df`라는 변수에 저장하면 이후 코드에서 데이터를 다시 불러오지 않고 계속 사용할 수 있다.

`df`는 `DataFrame`을 줄여서 관례적으로 사용하는 변수명일 뿐이므로, `titanic_data`처럼 다른 이름을 사용해도 된다.

**예시 코드**

```python
import pandas as pd

df = pd.read_csv(data_path)

print("데이터 크기:", df.shape)
print("컬럼:", df.columns.tolist())
display(df.head())
```

- `df.shape`: 행과 열의 개수를 확인한다.
- `df.columns.tolist()`: 전체 컬럼명을 확인한다.
- `df.head()`: 처음 5개 행을 확인한다.

---

##### 2. `merge`와 `concat`의 차이

**질문**

pandas의 `merge`와 `concat`은 데이터를 합치는 기준과 방식에서 어떤 차이가 있는가?

**답변**

`merge`는 공통된 ID나 키 값을 기준으로 관련 있는 행을 찾아 연결한다. SQL의 `JOIN`과 비슷하다.

`concat`은 공통 키를 찾아 연결하는 것이 아니라, 여러 DataFrame을 행 방향 또는 열 방향으로 이어 붙인다.

- `merge`: 같은 대상의 서로 다른 정보를 연결할 때 사용
- `concat(axis=0)`: 같은 구조의 데이터를 아래로 추가할 때 사용
- `concat(axis=1)`: 인덱스를 기준으로 데이터를 옆으로 붙일 때 사용

**예시 코드**

```python
import pandas as pd

students = pd.DataFrame({
    "id": [1, 2],
    "name": ["민수", "지수"]
})

scores = pd.DataFrame({
    "id": [1, 2],
    "score": [90, 85]
})

merged = pd.merge(students, scores, on="id")
display(merged)
```

**실행 결과**

```text
   id name  score
0   1   민수     90
1   2   지수     85
```

`id`가 같은 학생의 이름과 점수를 찾아 연결한다.

```python
january = pd.DataFrame({
    "name": ["민수", "지수"],
    "score": [90, 85]
})

february = pd.DataFrame({
    "name": ["철수", "영희"],
    "score": [70, 95]
})

concatenated = pd.concat(
    [january, february],
    ignore_index=True
)

display(concatenated)
```

**실행 결과**

```text
  name  score
0   민수     90
1   지수     85
2   철수     70
3   영희     95
```

두 DataFrame을 행 방향으로 이어 붙인다.

---

##### 3. 인코딩과 One-Hot Encoding

**질문**

인코딩은 무엇이며, `Embarked`처럼 순서가 없는 범주형 데이터를 왜 One-Hot Encoding으로 변환해야 하는가?

**답변**

인코딩은 문자나 범주로 구성된 데이터를 머신러닝 모델이 계산할 수 있는 숫자 형태로 변환하는 과정이다.

Titanic 데이터의 `Sex`와 `Embarked`는 문자형 범주이므로 대부분의 머신러닝 모델에 그대로 넣을 수 없다.

예를 들어 `Embarked`를 다음과 같이 변환하는 것은 적절하지 않을 수 있다.

- `C → 1`
- `Q → 2`
- `S → 3`

이렇게 변환하면 모델이 실제로 존재하지 않는 `S > Q > C`라는 순서와 크기 관계를 학습할 수 있기 때문이다.

One-Hot Encoding을 사용하면 각 범주를 별도의 0과 1 컬럼으로 표현하므로 잘못된 순서 관계가 생기지 않는다.

실제 모델링에서는 전체 데이터를 먼저 train과 test로 나눈 다음, train 데이터에서만 인코딩 기준을 학습해야 한다.

**예시 코드**

```python
import pandas as pd

sample = pd.DataFrame({
    "Embarked": ["S", "C", "Q"]
})

encoded = pd.get_dummies(
    sample,
    columns=["Embarked"],
    dtype=int
)

display(encoded)
```

**실행 결과**

```text
   Embarked_C  Embarked_Q  Embarked_S
0           0           0           1
1           1           0           0
2           0           1           0
```

실제 머신러닝 Pipeline에서는 다음과 같이 사용할 수 있다.

```python
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_encoded = encoder.fit_transform(X_train_cat)
X_test_encoded = encoder.transform(X_test_cat)
```

- train에는 `fit_transform()`을 사용한다.
- test에는 train에서 학습한 기준으로 `transform()`만 사용한다.

---

##### 4. p-value와 귀무가설

**질문**

p-value와 귀무가설은 무엇이며, 통계 검정 결과를 어떻게 판단해야 하는가?

**답변**

귀무가설은 일반적으로 두 집단 사이에 차이가 없다고 가정하는 출발점이다.

Titanic 데이터에서 생존자와 비생존자의 평균 운임을 비교한다면 가설은 다음과 같다.

- **귀무가설(H₀):** 생존자와 비생존자의 평균 운임에는 차이가 없다.
- **대립가설(H₁):** 생존자와 비생존자의 평균 운임에는 차이가 있다.

p-value는 귀무가설이 맞다고 가정했을 때, 현재 관찰된 것처럼 큰 차이가 우연히 나타날 가능성을 의미한다.

일반적으로 유의수준 `0.05`를 기준으로 판단한다.

- `p-value < 0.05`: 귀무가설을 기각하며, 통계적으로 유의한 차이가 있다고 판단한다.
- `p-value ≥ 0.05`: 귀무가설을 기각할 근거가 부족하다고 판단한다.

p-value는 귀무가설이 참일 확률이 아니며, 두 집단의 차이가 발생한 원인이나 차이의 실제 크기를 증명하지도 않는다.

**예시 코드**

```python
from scipy import stats

survived = df_work.loc[
    df_work["Survived"] == 1,
    "Fare"
]

not_survived = df_work.loc[
    df_work["Survived"] == 0,
    "Fare"
]

t_stat, p_value = stats.ttest_ind(
    survived,
    not_survived,
    equal_var=False
)

print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.3e}")

if p_value < 0.05:
    print("귀무가설 기각")
else:
    print("귀무가설을 기각하지 못함")
```

`equal_var=False`는 두 집단의 분산이 같다고 가정하지 않는 Welch의 t-검정을 사용한다는 의미다.

---

##### 5. 과적합·과소적합과 `train_test_split`

**질문**

과적합과 과소적합은 무엇이며, `train_test_split`을 이용해 이를 어떻게 확인할 수 있는가?

**답변**

**과적합(Overfitting)**은 모델이 학습 데이터의 일반적인 패턴뿐만 아니라 우연한 특징과 잡음까지 지나치게 학습한 상태다.

- train 성능은 매우 높다.
- test 성능은 상대적으로 낮다.
- 학습 데이터를 외웠지만 새로운 데이터에는 잘 적용되지 않는 상태다.

**과소적합(Underfitting)**은 모델이 너무 단순하거나 학습이 부족해 데이터의 기본적인 패턴도 충분히 학습하지 못한 상태다.

- train 성능이 낮다.
- test 성능도 낮다.
- 학습 데이터와 새로운 데이터 모두에서 성능이 좋지 않다.

`train_test_split`은 과적합을 직접 제거하지는 않는다. 다만 전체 데이터를 학습용과 평가용으로 나눠 train 성능과 test 성능을 비교할 수 있게 하므로 과적합을 발견하는 데 도움을 준다.

**예시 코드**

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model.fit(X_train, y_train)

train_score = model.score(X_train, y_train)
test_score = model.score(X_test, y_test)

print("Train accuracy:", train_score)
print("Test accuracy:", test_score)
print("성능 차이:", train_score - test_score)
```

**실행 결과 해석**

```text
Train accuracy: 0.98
Test accuracy: 0.75
```

train 성능만 매우 높고 test 성능이 크게 낮으므로 과적합을 의심할 수 있다.

```text
Train accuracy: 0.65
Test accuracy: 0.63
```

두 성능이 모두 낮으므로 과소적합을 의심할 수 있다.

```text
Train accuracy: 0.82
Test accuracy: 0.80
```

두 성능이 모두 비교적 높고 차이가 작다면 새로운 데이터에도 비교적 안정적으로 적용되는 모델이라고 볼 수 있다.

다만 정확도 차이 하나만으로 과적합을 단정하지 않고 교차검증, 데이터 크기, 모델 복잡도 등을 함께 확인해야 한다.

---

##### 6. 약 80% 정확도와 혼동행렬 해석

**질문**

모델의 약 80% 정확도와 혼동행렬의 TN 97, FP 13, FN 23, TP 46은 각각 무엇을 의미하는가?

**답변**

현재 혼동행렬은 다음과 같다.

```text
[[97 13]
 [23 46]]
```

scikit-learn의 이진 분류 혼동행렬은 다음 순서로 표시된다.

```text
[[TN FP]
 [FN TP]]
```

각 숫자의 의미는 다음과 같다.

- **TN 97명:** 실제 비생존자를 비생존으로 정확하게 예측
- **FP 13명:** 실제 비생존자를 생존으로 잘못 예측
- **FN 23명:** 실제 생존자를 비생존으로 잘못 예측
- **TP 46명:** 실제 생존자를 생존으로 정확하게 예측

전체 테스트 승객은 다음과 같이 179명이다.

```text
97 + 13 + 23 + 46 = 179
```

이 중 올바르게 예측한 승객은 TN과 TP를 합친 143명이다.

```text
97 + 46 = 143
```

따라서 정확도는 다음과 같다.

```text
Accuracy = (TN + TP) / 전체 인원
         = (97 + 46) / 179
         = 0.7989
         ≈ 79.9%
```

약 80% 정확도는 테스트 승객 179명 중 약 80%를 올바르게 분류했다는 뜻이다. 특정 승객의 생존 확률이 80%라는 의미는 아니다.

생존 클래스의 Recall은 실제 생존자 중 모델이 생존으로 찾아낸 비율이다.

```text
Recall = TP / (TP + FN)
       = 46 / (46 + 23)
       = 0.6667
```

즉, 실제 생존자는 총 69명이었고 모델은 그중 46명을 생존으로 찾았지만 23명을 비생존으로 잘못 예측했다.

**예시 코드**

```python
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
)

accuracy = accuracy_score(y_test, baseline_pred)
precision = precision_score(y_test, baseline_pred)
recall = recall_score(y_test, baseline_pred)
f1 = f1_score(y_test, baseline_pred)

cm = confusion_matrix(
    y_test,
    baseline_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = cm.ravel()

print("accuracy:", round(accuracy, 4))
print("precision:", round(precision, 4))
print("recall:", round(recall, 4))
print("f1:", round(f1, 4))
print(cm)
print({
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp,
})
```

**실행 결과**

```text
accuracy: 0.7989
precision: 0.7797
recall: 0.6667
f1: 0.7188

[[97 13]
 [23 46]]

{'TN': 97, 'FP': 13, 'FN': 23, 'TP': 46}
```

**최종 해석**

혼동행렬에서 97명은 실제 비생존자를 모델이 비생존으로 정확하게 예측한 경우(TN)이고, 13명은 실제 비생존자를 생존으로 잘못 예측한 경우(FP)이다. 또한 실제 생존자 23명을 비생존으로 잘못 예측했고(FN), 실제 생존자 46명을 생존으로 정확하게 예측했다(TP). 특히 생존자 중 23명을 놓쳤기 때문에 생존 클래스의 Recall은 약 0.667로 나타났다.
````